<a href="https://colab.research.google.com/github/gilbertoesp/galilean_moons/blob/master/Calculo_1_para_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cálculo



In [9]:
# ============================================================================
# CELDA 0 — Arquitectura base
# ============================================================================
# ¿Por qué cálculo para inteligencia artificial?
#
# Cada vez que un modelo "aprende", está resolviendo un problema de
# optimización: encontrar los parámetros que minimizan una función de error.
# La herramienta fundamental para eso es la DERIVADA.
#
# Esta notebook construye tu intuición visual desde los fundamentos
# —funciones, límites, continuidad— hasta la idea que lo conecta todo:
# la COMPOSICIÓN DE FUNCIONES y la regla de la cadena,
# que es exactamente lo que hace backpropagation.
#
# Empecemos construyendo las herramientas que usaremos en cada celda.
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from functools import wraps

# ── Estilo global consistente ──
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        12,
    "axes.titlesize":   14,
    "axes.labelsize":   12,
})

COLORES = {
    "funcion":   "#4C72B0",  # azul — función original
    "derivada":  "#DD8452",  # naranja — derivada
    "compuesta": "#55A868",  # verde — composición
    "auxiliar":  "#C44E52",  # rojo — anotaciones
    "sutil":     "#8C8C8C",  # gris — elementos secundarios
}


# ── Decorador: aplica estilo consistente a cualquier función de graficado ──
def estilo_calculo(titulo=""):
    """Decorador que envuelve una función de graficado con estilo uniforme.

    Uso:
        @estilo_calculo("Mi gráfica")
        def mi_grafica(ax, **params):
            ax.plot(x, y)

    El decorador se encarga de crear la figura, aplicar título,
    grid y layout.
    """
    def decorador(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            fig, axes = func(*args, **kwargs)
            if titulo:
                fig.suptitle(titulo, fontsize=15, fontweight="bold", y=1.02)
            plt.tight_layout()
            plt.show() # Re-agregar plt.show() para que el decorador controle la visualización
            plt.close(fig) # Cerrar la figura para evitar duplicidad de interact
            return None # Retornar None para que interact no intente mostrar nada más
        return wrapper
    return decorador


class Funcion:
    """Representa una función matemática con su derivada numérica.

    Filosofía: una función no es solo f(x) → y. Es un objeto con nombre,
    expresión, y la capacidad de calcular su propia derivada.

    Ejemplo:
        cuadrada = Funcion(lambda x: x**2, "x²")
        cuadrada(3)          # → 9
        cuadrada.derivada(3) # → 6.0 (aproximación numérica)
    """

    def __init__(self, f, nombre, color=None):
        self.f = f
        self.nombre = nombre
        self.color = color or COLORES["funcion"]

    def __call__(self, x):
        """Hace que el objeto sea invocable: funcion(x) en lugar de funcion.f(x)."""
        return self.f(x)

    def derivada(self, x, h=1e-7):
        """Derivada numérica por diferencias centrales: [f(x+h) - f(x-h)] / 2h.

        Las diferencias centrales son más precisas que las adelantadas
        porque los errores de orden par se cancelan. Es O(h²) vs O(h).
        """
        return (self.f(x + h) - self.f(x - h)) / (2 * h)

    def __matmul__(self, otra):
        """Composición de funciones con el operador @.

        Uso: (f @ g)(x) = f(g(x))

        Python nos permite sobrecargar @ (__matmul__), que resulta
        perfecto para composición: es visual y matemáticamente natural.
        """
        return Composicion(self, otra)

    def __repr__(self):
        return f"Funcion({self.nombre})"


class Composicion(Funcion):
    """Composición f(g(x)) con regla de la cadena incorporada.

    La derivada de f(g(x)) es f'(g(x)) · g'(x).
    Esto es EXACTAMENTE lo que hace backpropagation en cada capa.
    """

    def __init__(self, externa, interna):
        self.externa = externa
        self.interna = interna
        nombre = f"{externa.nombre}({interna.nombre})"
        super().__init__(
            f=lambda x: externa(interna(x)),
            nombre=nombre,
            color=COLORES["compuesta"],
        )

    def derivada(self, x, h=1e-7):
        """Regla de la cadena: f'(g(x)) · g'(x)."""
        return self.externa.derivada(self.interna(x), h) * self.interna.derivada(x, h)

    def componentes(self):
        """Devuelve las funciones que forman la composición."""
        return {"externa": self.externa, "interna": self.interna}


# ── Fábricas: funciones comunes listas para usar ──
def polinomio(coeficientes, nombre=None):
    """Crea una Funcion polinomial a partir de sus coeficientes.

    coeficientes: lista [a_n, a_{n-1}, ..., a_1, a_0]
    Usa np.polyval internamente, igual que numpy.
    """
    grado = len(coeficientes) - 1
    nombre = nombre or f"polinomio grado {grado}"
    return Funcion(lambda x, c=coeficientes: np.polyval(c, x), nombre)


def ciclica(tipo="sin", nombre=None):
    """Crea funciones trigonométricas básicas."""
    funciones = {"sin": np.sin, "cos": np.cos, "tan": np.tan}
    nombre = nombre or tipo
    return Funcion(funciones[tipo], nombre)


# ── Verificación rápida ──
_f = Funcion(lambda x: x**2, "x²")
assert abs(_f.derivada(3) - 6.0) < 1e-5, "La derivada numérica no funciona"

_g = ciclica("sin")
_compuesta = _f @ _g  # sin²(x)
assert isinstance(_compuesta, Composicion), "La composición no funciona"

print("✓ Arquitectura lista.")
print(f"  Ejemplo: f = {_f}, f(3) = {_f(3)}, f'(3) ≈ {_f.derivada(3):.4f}")
print(f"  Composición: {_compuesta}")

✓ Arquitectura lista.
  Ejemplo: f = Funcion(x²), f(3) = 9, f'(3) ≈ 6.0000
  Composición: Funcion(x²(sin))


## Funciones polinomiales

In [10]:
# ============================================================================
# CELDA 1 — Funciones polinomiales y sus derivadas
# ============================================================================
# ¿Por qué empezar con polinomios?
#
# Son las funciones más "domables" que existen: predecibles, diferenciables
# en todo punto, y fáciles de visualizar. En ML aparecen directamente en
# regresión polinomial, pero su importancia real es más profunda:
#
# El teorema de Taylor dice que CUALQUIER función suave se puede aproximar
# localmente por un polinomio. Cuando hacemos gradient descent, estamos
# confiando en esa aproximación: cerca del punto actual, la función de
# pérdida "se comporta como" un polinomio de grado bajo.
#
# Observa cómo la derivada siempre reduce el grado en 1.
# Un polinomio de grado n tiene una derivada de grado n-1.
# Eso significa que derivar SIMPLIFICA. En ML, eso es clave:
# podemos calcular gradientes de funciones complejas porque cada
# operación elemental tiene una derivada más simple que ella misma.
# ============================================================================

# ── Dominio de graficado ──
x = np.linspace(-3, 3, 500)

# ── Polinomios predefinidos por grado ──
# Cada entrada: (coeficientes, nombre legible, nombre de la derivada)
POLINOMIOS = {
    1: {
        "f": polinomio([2, 1],    "2x + 1"),
        "desc": "Lineal: f(x) = 2x + 1 → f'(x) = 2 (constante)",
    },
    2: {
        "f": polinomio([1, -2, 1], "x² − 2x + 1"),
        "desc": "Cuadrática: f(x) = x² − 2x + 1 → f'(x) = 2x − 2 (lineal)",
    },
    3: {
        "f": polinomio([1, 0, -3, 0], "x³ − 3x"),
        "desc": "Cúbica: f(x) = x³ − 3x → f'(x) = 3x² − 3 (cuadrática)",
    },
    4: {
        "f": polinomio([0.5, 0, -3, 0, 2], "0.5x⁴ − 3x² + 2"),
        "desc": "Grado 4: máximos y mínimos locales donde f'(x) = 0",
    },
    5: {
        "f": polinomio([0.1, 0, -1, 0, 2, 0], "0.1x⁵ − x³ + 2x"),
        "desc": "Grado 5: la derivada es grado 4 → hasta 4 raíces reales",
    },
}


@estilo_calculo("")
def graficar_polinomio(grado):
    """Muestra un polinomio y su derivada lado a lado."""
    datos = POLINOMIOS[grado]
    f = datos["f"]

    # Evaluar función y derivada sobre el dominio
    y_f = f(x)
    y_d = np.array([f.derivada(xi) for xi in x])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # ── Panel izquierdo: la función ──
    ax1.plot(x, y_f, color=COLORES["funcion"], lw=2.5, label=f"f(x) = {f.nombre}")
    ax1.axhline(y=0, color=COLORES["sutil"], lw=0.8, ls="--")
    ax1.set_title("f(x) — Función original")
    ax1.set_xlabel("x")
    ax1.set_ylabel("f(x)")
    ax1.legend(loc="upper left", fontsize=11)
    ax1.set_ylim(-10, 10)

    # ── Panel derecho: la derivada ──
    ax2.plot(x, y_d, color=COLORES["derivada"], lw=2.5, label=f"f'(x)")
    ax2.axhline(y=0, color=COLORES["sutil"], lw=0.8, ls="--")

    # Marcar los ceros de la derivada (puntos críticos)
    # Buscamos cambios de signo en y_d como aproximación a las raíces
    cambios = np.where(np.diff(np.sign(y_d)))[0]
    for idx in cambios:
        x_crit = x[idx]
        y_crit = f(x_crit)
        ax2.plot(x_crit, 0, "o", color=COLORES["auxiliar"], ms=8, zorder=5)
        # Marcar el punto correspondiente en la función original
        ax1.plot(x_crit, y_crit, "o", color=COLORES["auxiliar"], ms=8, zorder=5)
        ax1.annotate(
            f"x={x_crit:.1f}",
            (x_crit, y_crit),
            textcoords="offset points",
            xytext=(10, 10),
            fontsize=9,
            color=COLORES["auxiliar"],
            fontweight="bold",
        )

    ax2.set_title("f'(x) — Derivada (grado reducido en 1)")
    ax2.set_xlabel("x")
    ax2.set_ylabel("f'(x)")
    ax2.legend(loc="upper left", fontsize=11)
    ax2.set_ylim(-10, 10)

    # ── Anotación descriptiva ──
    fig.text(
        0.5, -0.05, datos["desc"],
        ha="center", fontsize=12, style="italic", color=COLORES["sutil"],
    )

    return fig, (ax1, ax2)


# ── Lanzar visualización interactiva ──
interact(
    graficar_polinomio,
    grado=IntSlider(min=1, max=5, step=1, value=2, description="Grado:"),
)

interactive(children=(IntSlider(value=2, description='Grado:', max=5, min=1), Output()), _dom_classes=('widget…

<function __main__.graficar_polinomio(grado)>

## Funciones ciclicas

In [11]:
# ============================================================================
# CELDA 2 — Funciones cíclicas: seno, coseno y sus derivadas
# ============================================================================
# ¿Por qué funciones periódicas en ML?
#
# Aparecen en más lugares de los que imaginas:
#   1. Codificación posicional en Transformers — sin(pos / 10000^(2i/d))
#      Cada dimensión del embedding usa senos y cosenos de frecuencias
#      distintas para que el modelo "sepa" la posición de cada token.
#   2. Learning rate schedulers cíclicos (cosine annealing) —
#      lr(t) = lr_min + 0.5(lr_max - lr_min)(1 + cos(πt/T))
#   3. Transformadas de Fourier — descomponer señales en frecuencias.
#
# La propiedad mágica: derivar sin(x) produce cos(x), que es el mismo
# sin(x) desplazado π/2. Derivar no cambia la FORMA, solo la FASE.
# Esto hace que las funciones cíclicas sean numéricamente estables
# durante backpropagation: sus gradientes nunca explotan ni desaparecen.
#
# Mueve los sliders para desarrollar intuición sobre tres parámetros:
#   A (amplitud) → escala vertical, amplifica el gradiente
#   ω (frecuencia) → comprime horizontalmente, MULTIPLICA el gradiente
#   φ (fase) → desplaza horizontalmente, no afecta la magnitud
# ============================================================================

x = np.linspace(-2 * np.pi, 2 * np.pi, 600)


def onda_parametrica(A, omega, phi):
    """Construye f(x) = A·sin(ωx + φ) como objeto Funcion.

    La derivada analítica es f'(x) = A·ω·cos(ωx + φ).
    Nota cómo ω aparece como FACTOR en la derivada:
    mayor frecuencia → mayor gradiente. En redes neuronales,
    esto es análogo a cómo los pesos escalan los gradientes.
    """
    nombre = f"{A}·sin({omega}x + {phi:.1f})"
    return Funcion(
        f=lambda x, A=A, w=omega, p=phi: A * np.sin(w * x + p),
        nombre=nombre,
        color=COLORES["funcion"],
    )


@estilo_calculo("")
def graficar_ciclica(
    A, omega, phi, mostrar,
):
    """Visualiza f(x) = A·sin(ωx + φ) y su derivada."""
    f = onda_parametrica(A, omega, phi)

    y_f = f(x)
    y_d = np.array([f.derivada(xi) for xi in x])

    # ── Derivada analítica para comparar ──
    y_d_analitica = A * omega * np.cos(omega * x + phi)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # ── Panel izquierdo: la función ──
    if mostrar in ("f(x)", "Ambas"):
        ax1.plot(x, y_f, color=COLORES["funcion"], lw=2.5,
                 label=f"f(x) = {A}·sin({omega}x + {phi:.1f})")
    if mostrar in ("f'(x)", "Ambas"):
        ax1.plot(x, y_d, color=COLORES["derivada"], lw=2.5, ls="--",
                 label=f"f'(x) = {A}·{omega}·cos({omega}x + {phi:.1f})")

    ax1.axhline(y=0, color=COLORES["sutil"], lw=0.8, ls="--")
    ax1.set_title("Función y derivada superpuestas")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.legend(loc="upper right", fontsize=10)
    ax1.set_ylim(-6, 6)

    # ── Panel derecho: anatomía del gradiente ──
    # Mostrar cómo A y ω afectan la magnitud del gradiente
    ax2.fill_between(x, -abs(A * omega), abs(A * omega),
                     alpha=0.1, color=COLORES["derivada"],
                     label=f"Magnitud máx |f'| = A·ω = {abs(A*omega):.1f}")
    ax2.plot(x, y_d_analitica, color=COLORES["derivada"], lw=2,
             label="f'(x) analítica")

    # Marcar el desfase de π/2 entre sin y cos
    if mostrar == "Ambas" and omega > 0:
        # Primer máximo de f(x): en x donde ωx + φ = π/2
        x_max_f = (np.pi / 2 - phi) / omega
        x_cero_d = x_max_f  # f' cruza cero donde f tiene máximo
        if -2 * np.pi < x_max_f < 2 * np.pi:
            ax2.axvline(x=x_cero_d, color=COLORES["auxiliar"],
                        ls=":", lw=1.5, alpha=0.7)
            ax2.annotate(
                "f máx → f' = 0",
                (x_cero_d, 0),
                textcoords="offset points",
                xytext=(15, A * omega * 0.6),
                fontsize=10, color=COLORES["auxiliar"],
                arrowprops=dict(arrowstyle="->", color=COLORES["auxiliar"]),
            )

    ax2.axhline(y=0, color=COLORES["sutil"], lw=0.8, ls="--")
    ax2.set_title("Anatomía del gradiente")
    ax2.set_xlabel("x")
    ax2.set_ylabel("f'(x)")
    ax2.legend(loc="upper right", fontsize=10)
    ax2.set_ylim(-6, 6)

    # ── Nota al pie ──
    nota = (
        f"Amplitud A={A} escala f y f'. "
        f"Frecuencia ω={omega} MULTIPLICA el gradiente (A·ω={A*omega:.1f}). "
        f"Fase φ={phi:.1f} desplaza sin cambiar magnitud."
    )
    fig.text(0.5, -0.06, nota, ha="center", fontsize=11,
             style="italic", color=COLORES["sutil"])

    return fig, (ax1, ax2)


# ── Lanzar visualización interactiva ──
interact(
    graficar_ciclica,
    A=FloatSlider(min=0.5, max=4.0, step=0.5, value=1.0,
                  description="A (amplitud):"),
    omega=FloatSlider(min=0.5, max=4.0, step=0.5, value=1.0,
                      description="ω (frecuencia):"),
    phi=FloatSlider(min=0, max=2 * np.pi, step=0.1, value=0,
                    description="φ (fase):"),
    mostrar=Dropdown(options=["f(x)", "f'(x)", "Ambas"], value="Ambas",
                     description="Mostrar:"),
)

interactive(children=(FloatSlider(value=1.0, description='A (amplitud):', max=4.0, min=0.5, step=0.5), FloatSl…

<function __main__.graficar_ciclica(A, omega, phi, mostrar)>

## Limites

El panel de convergencia (derecho) es el diferenciador. Muchas visualizaciones de límites solo muestran la función con un punto que se acerca. El panel de convergencia con escala logarítmica en |x − a| le da al estudiante una segunda representación: conforme nos movemos a la derecha (distancia decrece), ¿convergen las líneas naranja y verde al mismo valor? Si sí, el límite existe. Si no, no existe. Es la definición epsilon-delta hecha visual.

El catálogo de cuatro ejemplos está ordenado con intención. Los dos primeros (sin(x)/x y (x²−4)/(x−2)) son indeterminaciones 0/0 donde el límite sí existe, el tercero (|x|/x) rompe esa expectativa mostrando que el límite puede no existir, y el cuarto (eˣ−1)/x cierra el arco conectando con la exponencial. Además, el caso (x²−4)/(x−2) siembra la semilla para la Celda 6 donde racionalizaremos esa indeterminación algebraicamente.

El círculo abierto rojo en el punto de interés refuerza la notación estándar de "este valor no está definido", que muchos estudiantes ven en libros pero pocas veces en código.

In [12]:
# ============================================================================
# CELDA 3 — Límites: la pregunta que lo inicia todo
# ============================================================================
# ¿Qué pasa cuando nos ACERCAMOS a un punto sin tocarlo?
#
# Esa pregunta es el cálculo entero en una línea.
# La derivada es un límite: f'(x) = lim[h→0] (f(x+h) - f(x)) / h
# La integral es un límite de sumas. Incluso las redes neuronales
# dependen de que ciertos límites existan y sean finitos.
#
# En esta celda vas a VER el límite en acción:
#   - Un punto se acerca desde la izquierda y la derecha.
#   - Si ambos lados convergen al mismo valor, el límite existe.
#   - Si no, el límite no existe (y la derivada tampoco).
#
# ¿Por qué importa en ML?
# Gradient descent calcula f'(x) = lim[h→0] (f(x+h) - f(x)) / h.
# Si ese límite no existe en algún punto, el optimizador no sabe
# hacia dónde moverse. Por eso funciones como ReLU necesitan
# convenciones especiales en x=0, y por eso se inventaron
# alternativas suaves como SiLU y GELU.
# ============================================================================

x = np.linspace(-4, 4, 1000)

# ── Catálogo de funciones con límites interesantes ──
# Cada entrada define: función, punto de interés, valor del límite,
# y una descripción de qué ocurre.

LIMITES = {
    "sin(x)/x → 0": {
        "f": Funcion(
            lambda x: np.where(np.abs(x) < 1e-12, 1.0, np.sin(x) / x),
            "sin(x)/x",
        ),
        "punto": 0.0,
        "limite": 1.0,
        "dominio": (-4 * np.pi, 4 * np.pi),
        "desc": (
            "El límite más famoso del cálculo. f(0) no está definida "
            "(0/0), pero lim[x→0] sin(x)/x = 1. "
            "Esta es la base para derivar sin(x)."
        ),
    },
    "(x²−4)/(x−2) → 2": {
        "f": Funcion(
            lambda x: np.where(np.abs(x - 2) < 1e-12, 4.0, (x**2 - 4) / (x - 2)),
            "(x²−4)/(x−2)",
        ),
        "punto": 2.0,
        "limite": 4.0,
        "dominio": (-1, 5),
        "desc": (
            "Indeterminación 0/0. Algebraicamente: (x²−4)/(x−2) = x+2. "
            "El límite es 4, aunque f(2) no existe. "
            "En la Celda 6 veremos cómo racionalizar esto."
        ),
    },
    "|x|/x → 0": {
        "f": Funcion(
            lambda x: np.where(np.abs(x) < 1e-12, np.nan, np.abs(x) / x),
            "|x|/x",
        ),
        "punto": 0.0,
        "limite": None,  # No existe
        "dominio": (-3, 3),
        "desc": (
            "¡El límite NO existe! Desde la izquierda → −1, "
            "desde la derecha → +1. Esto es análogo a la derivada "
            "de |x| en x=0: no está definida. ReLU hereda este problema."
        ),
    },
    "(eˣ−1)/x → 0": {
        "f": Funcion(
            lambda x: np.where(np.abs(x) < 1e-12, 1.0, (np.exp(x) - 1) / x),
            "(eˣ−1)/x",
        ),
        "punto": 0.0,
        "limite": 1.0,
        "dominio": (-4, 4),
        "desc": (
            "Otro 0/0. lim[x→0] (eˣ−1)/x = 1. "
            "Este límite es la derivada de eˣ evaluada en x=0. "
            "La exponencial es su propia derivada: d/dx[eˣ] = eˣ."
        ),
    },
}


def _evaluar_acercamiento(f, punto, epsilon):
    """Genera puntos acercándose al límite desde ambos lados.

    Devuelve arrays con las posiciones y valores de f(x)
    para visualizar la convergencia.
    """
    # Puntos que se acercan desde la izquierda y la derecha
    n_puntos = 12
    distancias = np.logspace(0, np.log10(epsilon), n_puntos)

    x_izq = punto - distancias[::-1]
    x_der = punto + distancias

    with np.errstate(divide="ignore", invalid="ignore"):
        y_izq = np.array([f(xi) for xi in x_izq])
        y_der = np.array([f(xi) for xi in x_der])

    return x_izq, y_izq, x_der, y_der


@estilo_calculo("")
def graficar_limite(ejemplo, epsilon):
    """Visualiza el comportamiento de f(x) cuando x se acerca a un punto."""
    datos = LIMITES[ejemplo]
    f = datos["f"]
    punto = datos["punto"]
    limite = datos["limite"]
    a, b = datos["dominio"]

    # Dominio de graficado adaptado a cada ejemplo
    x_plot = np.linspace(a, b, 800)
    with np.errstate(divide="ignore", invalid="ignore"):
        y_plot = f(x_plot)

    # Puntos acercándose al límite
    x_izq, y_izq, x_der, y_der = _evaluar_acercamiento(f, punto, epsilon)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                                   gridspec_kw={"width_ratios": [2, 1]})

    # ══════════════════════════════════════════════════════════
    # Panel izquierdo: la función completa con zoom al punto
    # ══════════════════════════════════════════════════════════
    y_safe = np.where(np.isfinite(y_plot), y_plot, np.nan)
    ax1.plot(x_plot, y_safe, color=COLORES["funcion"], lw=2, label=f.nombre, zorder=2)

    # Puntos acercándose (los que están en el rango visible)
    ax1.scatter(x_izq, y_izq, color=COLORES["derivada"], s=40, zorder=4,
                label="Desde la izquierda", edgecolors="white", linewidth=0.5)
    ax1.scatter(x_der, y_der, color=COLORES["compuesta"], s=40, zorder=4,
                label="Desde la derecha", edgecolors="white", linewidth=0.5)

    # Marcar el punto problemático con un círculo abierto
    ax1.plot(punto, limite if limite is not None else 0, "o",
             color=COLORES["auxiliar"], ms=12, mfc="white", mew=2.5, zorder=5)

    # Línea horizontal del valor del límite
    if limite is not None:
        ax1.axhline(y=limite, color=COLORES["auxiliar"], ls=":", lw=1.5, alpha=0.6,
                    label=f"L = {limite}")

    ax1.axvline(x=punto, color=COLORES["sutil"], ls=":", lw=1, alpha=0.5)
    ax1.set_title(f"f(x) = {f.nombre}")
    ax1.set_xlabel("x")
    ax1.set_ylabel("f(x)")
    ax1.legend(loc="best", fontsize=9)

    y_finite = y_safe[np.isfinite(y_safe)]
    if len(y_finite) > 0:
        y_range = max(abs(y_finite.min()), abs(y_finite.max()), 2)
        ax1.set_ylim(-y_range * 1.2, y_range * 1.2)

    # ══════════════════════════════════════════════════════════
    # Panel derecho: convergencia numérica
    # ══════════════════════════════════════════════════════════
    # Mostrar los valores de f(x) vs distancia al punto
    dist_izq = np.abs(x_izq - punto)
    dist_der = np.abs(x_der - punto)

    ax2.plot(dist_izq, y_izq, "o-", color=COLORES["derivada"], lw=1.5,
             ms=6, label="Izquierda")
    ax2.plot(dist_der, y_der, "s-", color=COLORES["compuesta"], lw=1.5,
             ms=6, label="Derecha")

    if limite is not None:
        ax2.axhline(y=limite, color=COLORES["auxiliar"], ls="--", lw=2,
                    label=f"Límite = {limite}", alpha=0.8)

    ax2.set_xscale("log")
    ax2.set_title("Convergencia: f(x) vs |x − a|")
    ax2.set_xlabel("|x − a|  (escala log)")
    ax2.set_ylabel("f(x)")
    ax2.legend(loc="best", fontsize=9)
    ax2.invert_xaxis()  # Más cerca del punto → derecha

    # ── Nota al pie ──
    fig.text(0.5, -0.08, datos["desc"], ha="center", fontsize=10,
             style="italic", color=COLORES["sutil"], wrap=True)

    return fig, (ax1, ax2)


# ── Lanzar visualización interactiva ──
interact(
    graficar_limite,
    ejemplo=Dropdown(
        options=list(LIMITES.keys()),
        value="sin(x)/x → 0",
        description="Función:",
    ),
    epsilon=FloatSlider(
        min=0.001, max=2.0, step=0.001, value=0.5,
        description="ε (cercanía):",
        readout_format=".3f",
    ),
)

interactive(children=(Dropdown(description='Función:', options=('sin(x)/x → 0', '(x²−4)/(x−2) → 2', '|x|/x → 0…

<function __main__.graficar_limite(ejemplo, epsilon)>